In [ ]:
# ============================================================
# MESA V3 — CELL 1
# Install and import all dependencies
# ============================================================
!pip install mesa torch torchvision -q

from google.colab import drive
drive.mount('/content/drive')

import mesa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import numpy as np
import json
import pickle
import warnings
from collections import defaultdict, deque
from sklearn.preprocessing import MinMaxScaler
warnings.filterwarnings('ignore')

DRIVE_PATH = '/content/drive/My Drive/Colab Notebooks/V3/'

print(f"Mesa version:   {mesa.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

DEVICE = torch.device('cuda' if torch.cuda.is_available()
                       else 'cpu')
print(f"Using device:   {DEVICE}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.1/275.1 kB 9.9 MB/s eta 0:00:00
Mounted at /content/drive
Mesa version:   3.5.1
PyTorch version: 2.11.0+cpu
CUDA available: False
Using device:   cpu


In [ ]:
# ============================================================
# CELL 2 — Load all pre-work outputs
# Every file produced in Stages 1-3 and Layer 2
# ============================================================

# Master price data
df = pd.read_csv(
    DRIVE_PATH + 'V3_Master_Wide.csv',
    parse_dates=['date']
)
df = df.sort_values(['city', 'date']).reset_index(drop=True)

# Stage 1 — embedding params (LSTM sequence lengths)
embedding_params = pd.read_csv(
    DRIVE_PATH + 'S1_embedding_params_full.csv'
)

# Stage 2 — confirmed causal variables
with open(DRIVE_PATH + 'S2_confirmed_causes.json') as f:
    confirmed_causes = json.load(f)

# Stage 2b — CTDE weights (VAR coefficients)
with open(DRIVE_PATH + 'S2b_ctde_weights.json') as f:
    ctde_weights = json.load(f)

# Stage 3 — scalers for runtime normalisation
with open(DRIVE_PATH + 'S3_scalers.json') as f:
    scaler_params = json.load(f)

# Stage 3 — phase state per city per month
phase_state = pd.read_csv(
    DRIVE_PATH + 'S3_phase_state.csv',
    parse_dates=['date']
)

# Layer 2 — monthly network metrics per node
network_metrics = pd.read_csv(
    DRIVE_PATH + 'L2_monthly_network_metrics.csv',
    parse_dates=['date']
)

CITIES = sorted(df['city'].unique())

CITY_INDICATORS = [
    'Rice', 'Salt', 'Seasoning', 'Curry', 'Thyme',
    'Red_Peppers', 'Tomatoes_Fresh', 'Veg_Oil',
    'Tinned_Tomatoes', 'Beef', 'Chicken', 'Onions',
    'PMS_Price', 'Bus_Intercity'
]
NATIONAL_INDICATORS = ['USD_NGN', 'CBN_MPR']
ALL_INDICATORS = CITY_INDICATORS + NATIONAL_INDICATORS

print(f"Master CSV:      {df.shape}")
print(f"Cities:          {CITIES}")
print(f"Phase state:     {phase_state.shape}")
print(f"Network metrics: {network_metrics.shape}")
print(f"Simulation:      Jan 2024 to Jun 2026 (Phase 1)")
print(f"Forecast:        Jul 2026 to Dec 2027 (Phase 2)")

Master CSV:      (540, 18)
Cities:          ['Abuja', 'Bauchi', 'Kano', 'Lagos', 'Onitsha', 'Port_Harcourt']
Phase state:     (419, 11)
Network metrics: (2580, 12)
Simulation:      Jan 2024 to Jun 2026 (Phase 1)
Forecast:        Jul 2026 to Dec 2027 (Phase 2)


In [ ]:
# ============================================================
# CELL 3 — Build runtime scalers from saved parameters
# Reconstruct MinMaxScaler per city from Stage 3 outputs
# Used to scale price observations before entering networks
# ============================================================

city_scalers = {}

for city in CITIES:
    if city not in scaler_params:
        continue
    params    = scaler_params[city]
    variables = params['variables']
    data_min  = np.array(params['data_min'])
    data_max  = np.array(params['data_max'])

    scaler = MinMaxScaler()
    # Reconstruct fitted scaler from saved min/max
    scaler.fit(np.vstack([data_min, data_max]))
    scaler.data_min_  = data_min
    scaler.data_max_  = data_max
    scaler.data_range_ = data_max - data_min
    scaler.scale_     = 1.0 / np.where(
        scaler.data_range_ == 0, 1, scaler.data_range_
    )
    scaler.min_       = -data_min * scaler.scale_
    scaler.n_features_in_ = len(variables)

    city_scalers[city] = {
        'scaler':    scaler,
        'variables': variables
    }

print("Runtime scalers built:")
for city, s in city_scalers.items():
    print(f"  {city:<15} {len(s['variables'])} variables")


def scale_observation(city, price_dict):
    """
    Scale a dict of {indicator: price} for a city
    using the fitted MinMaxScaler.
    Returns normalised numpy array.
    """
    if city not in city_scalers:
        return np.zeros(len(ALL_INDICATORS))

    scaler    = city_scalers[city]['scaler']
    variables = city_scalers[city]['variables']
    raw       = np.array([
        price_dict.get(v, 0.0) for v in variables
    ]).reshape(1, -1)

    try:
        scaled = scaler.transform(raw)[0]
    except Exception:
        scaled = np.clip(raw[0], 0, 1)

    return scaled.astype(np.float32)

Runtime scalers built:
  Abuja           16 variables
  Bauchi          16 variables
  Kano            16 variables
  Lagos           16 variables
  Onitsha         16 variables
  Port_Harcourt   14 variables


In [ ]:
# ============================================================
# CELL 4 — Build LSTM sequence lengths per city
# From Stage 1 Takens embedding τ and d values
# Each city's LSTM looks back a different number of months
# ============================================================

CITY_LSTM_PARAMS = {}

for city in CITIES:
    city_params = embedding_params[
        embedding_params['city'] == city
    ]
    if city_params.empty:
        CITY_LSTM_PARAMS[city] = {'seq_len': 12, 'hidden': 32}
        continue

    max_tau = int(city_params['tau'].max())
    max_d   = int(city_params['d'].max())

    # Sequence length = τ × (d - 1) — from Takens theory
    # Capped at 30 for Lagos (embedding point limitation)
    seq_len     = max_tau * max(1, max_d - 1)
    seq_len     = min(seq_len, 30)
    seq_len     = max(seq_len, 6)

    # LSTM hidden size scales with embedding complexity
    hidden_size = max(32, max_d * 8)

    CITY_LSTM_PARAMS[city] = {
        'seq_len':    seq_len,
        'hidden':     hidden_size,
        'max_tau':    max_tau,
        'max_d':      max_d
    }

print("LSTM parameters per city:")
print(f"{'City':<15} {'seq_len':<10} {'hidden':<10} "
      f"{'max_τ':<8} {'max_d'}")
print("-" * 55)
for city, p in CITY_LSTM_PARAMS.items():
    print(f"  {city:<13} {p['seq_len']:<10} {p['hidden']:<10} "
          f"{p['max_tau']:<8} {p['max_d']}")

LSTM parameters per city:
City            seq_len    hidden     max_τ    max_d
-------------------------------------------------------
  Abuja         22         32         11       3
  Bauchi        30         32         11       4
  Kano          16         32         8        3
  Lagos         30         48         10       6
  Onitsha       22         32         11       3
  Port_Harcourt 22         32         11       3


In [ ]:
# ============================================================
# CELL 5 — Neural network definitions (CORRECTED)
# a2c_update uses fresh forward pass for gradients
# ============================================================

class LSTMEncoder(nn.Module):
    def __init__(self, n_features, hidden_size, n_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            dropout=0.0
        )
        self.hidden_size = hidden_size

    def forward(self, x, hidden=None):
        out, (h_n, c_n) = self.lstm(x, hidden)
        return out[:, -1, :], (h_n, c_n)


class ActorNetwork(nn.Module):
    def __init__(self, lstm_hidden, phase_dim,
                 network_dim, n_actions):
        super().__init__()
        input_dim = lstm_hidden + phase_dim + network_dim
        self.fc1  = nn.Linear(input_dim, 64)
        self.fc2  = nn.Linear(64, 32)
        self.fc3  = nn.Linear(32, n_actions)

    def forward(self, lstm_out, phase_state, network_state):
        x = torch.cat([lstm_out, phase_state,
                        network_state], dim=-1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return F.softmax(self.fc3(x), dim=-1)


class CriticNetwork(nn.Module):
    def __init__(self, lstm_hidden, phase_dim, network_dim):
        super().__init__()
        input_dim = lstm_hidden + phase_dim + network_dim
        self.fc1  = nn.Linear(input_dim, 64)
        self.fc2  = nn.Linear(64, 32)
        self.fc3  = nn.Linear(32, 1)

    def forward(self, lstm_out, phase_state, network_state):
        x = torch.cat([lstm_out, phase_state,
                        network_state], dim=-1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class A2CAgent(nn.Module):
    def __init__(self, n_features, seq_len, hidden_size,
                 n_actions, phase_dim=4, network_dim=4,
                 lr=1e-3):
        super().__init__()
        self.encoder   = LSTMEncoder(n_features, hidden_size)
        self.actor     = ActorNetwork(
            hidden_size, phase_dim, network_dim, n_actions
        )
        self.critic    = CriticNetwork(
            hidden_size, phase_dim, network_dim
        )
        self.optimiser = optim.Adam(self.parameters(), lr=lr)
        self.hidden    = None

    def forward(self, price_seq, phase_obs, network_obs):
        lstm_out, self.hidden = self.encoder(
            price_seq, self.hidden
        )
        probs = self.actor(lstm_out, phase_obs, network_obs)
        value = self.critic(lstm_out, phase_obs, network_obs)
        return probs, value

    def reset_hidden(self):
        self.hidden = None

    def detach_hidden(self):
        if self.hidden is not None:
            self.hidden = (
                self.hidden[0].detach(),
                self.hidden[1].detach()
            )


def a2c_update(network, obs_tensors, action,
               reward, gamma=0.95, entropy_coef=0.01):
    """
    A2C update with fresh forward pass to ensure gradients exist.
    Separates inference (no_grad in agent.step) from training.
    obs_tensors: (price_seq, phase_obs, network_obs) stored
                 during agent.step() as detached clones
    action:      integer action index chosen during inference
    reward:      float reward from execute_action()
    """
    price_seq, phase_obs, network_obs = obs_tensors

    # Fresh forward pass WITH gradient tracking
    network.reset_hidden()
    probs, value = network(price_seq, phase_obs, network_obs)

    dist     = torch.distributions.Categorical(probs)
    log_prob = dist.log_prob(
        torch.tensor(action).to(DEVICE)
    )
    entropy  = dist.entropy()

    # Bootstrap next value estimate — no grad needed
    with torch.no_grad():
        network.reset_hidden()
        _, next_value = network(
            price_seq, phase_obs, network_obs
        )

    reward_t    = torch.tensor(
        [[reward]], dtype=torch.float32
    ).to(DEVICE)

    advantage   = reward_t + gamma * next_value - value
    actor_loss  = -log_prob * advantage.detach()
    critic_loss = advantage.pow(2)
    total_loss  = (actor_loss
                   + 0.5 * critic_loss
                   - entropy_coef * entropy)

    network.optimiser.zero_grad()
    total_loss.mean().backward()
    torch.nn.utils.clip_grad_norm_(
        network.parameters(), max_norm=1.0
    )
    network.optimiser.step()
    network.detach_hidden()

    return float(total_loss.mean().detach())


PHASE_DIM   = 4
NETWORK_DIM = 4

print("Cell 5 complete:")
print("  LSTMEncoder, ActorNetwork, CriticNetwork defined")
print("  A2CAgent — combined module with optimiser")
print("  a2c_update — fresh forward pass, gradients guaranteed")
print(f"  Phase state dim:   {PHASE_DIM}")
print(f"  Network state dim: {NETWORK_DIM}")

Cell 5 complete:
  LSTMEncoder, ActorNetwork, CriticNetwork defined
  A2CAgent — combined module with optimiser
  a2c_update — fresh forward pass, gradients guaranteed
  Phase state dim:   4
  Network state dim: 4


In [ ]:
# ============================================================
# CELL 6 — CTDE weight initialisation
# Seed A2C network weights from VAR coefficients
# This replaces random initialisation with historically
# grounded priors — agents start informed not blind
# ============================================================

def initialise_ctde_weights(network, city, breed,
                              ctde_weights):
    """
    Initialise Actor fc1 layer weights using VAR
    coefficients from Stage 2b.
    breed: 'trader' | 'transporter' | 'buyer' | 'banking'
    """
    city_weights = ctde_weights.get(city)
    if city_weights is None:
        return

    coef_tensor = city_weights.get('coef_tensor', {})
    variables   = city_weights.get('variables', [])

    if not coef_tensor or not variables:
        return

    # Choose target equations by breed
    if breed == 'trader':
        target_vars = [v for v in variables if v in [
            'Tomatoes_Fresh', 'Rice', 'Beef',
            'Chicken', 'Onions', 'Veg_Oil'
        ]]
    elif breed == 'transporter':
        target_vars = [v for v in variables if v in [
            'Bus_Intercity', 'PMS_Price'
        ]]
    elif breed == 'buyer':
        target_vars = [v for v in variables if v in [
            'Tomatoes_Fresh', 'Rice', 'Chicken', 'Beef'
        ]]
    elif breed == 'banking':
        target_vars = [v for v in variables if v in [
            'CBN_MPR', 'PMS_Price', 'USD_NGN'
        ]]
    else:
        return

    # Build coefficient vector from VAR lag-1 values
    coef_vector = []
    for effect in target_vars:
        if effect not in coef_tensor:
            continue
        for cause in variables:
            if cause in coef_tensor.get(effect, {}):
                lag_coefs = coef_tensor[effect][cause]
                coef_vector.append(
                    float(lag_coefs[0]) if lag_coefs else 0.0
                )

    if not coef_vector:
        return

    # Scale to [-0.1, 0.1] for stable initialisation
    coef_arr = np.array(coef_vector, dtype=np.float32)
    max_abs  = np.abs(coef_arr).max()
    if max_abs > 0:
        coef_arr = coef_arr / max_abs * 0.1

    # Apply to first layer of Actor — partial initialisation
    with torch.no_grad():
        fc1_weight = network.actor.fc1.weight
        n_rows     = min(fc1_weight.shape[0], len(coef_arr))
        for i in range(n_rows):
            fc1_weight[i, 0] = float(coef_arr[i])

    return len(coef_arr)


print("CTDE initialisation function defined.")
print("VAR coefficients seed Actor fc1 layer weights.")
print("Breeds: trader | transporter | buyer | banking")

CTDE initialisation function defined.
VAR coefficients seed Actor fc1 layer weights.
Breeds: trader | transporter | buyer | banking


In [ ]:
# ============================================================
# CELL 7 — All agent classes + E&K intermediary model
# FINAL VERSION with away trader return fix
# Trader goes to source city → buys at producer price
# Returns to HOME city next month → sells at home price
# This is the E&K Chapter 11 intermediary mechanism
# ============================================================

PRODUCER_PRICE_RATIO = {
    'Tomatoes_Fresh':  0.75,
    'Rice':            0.82,
    'Beef':            0.80,
    'Chicken':         0.78,
    'Onions':          0.77,
    'Veg_Oil':         0.85,
    'Seasoning':       0.83,
    'Curry':           0.83,
    'Thyme':           0.83,
    'Red_Peppers':     0.76,
    'Tinned_Tomatoes': 0.84,
    'Salt':            0.88,
    'default':         0.80,
}


def get_producer_price(model, city, commodity):
    """
    Farmgate/wholesale price traders pay to source goods.
    NBS market price = consumer price.
    Producer price = NBS price × ratio (below consumer price).
    Trader gross margin = consumer price - producer price.
    """
    market_price = model.get_price(city, commodity)
    ratio        = PRODUCER_PRICE_RATIO.get(
        commodity, PRODUCER_PRICE_RATIO['default']
    )
    return market_price * ratio


class TraderAgent(mesa.Agent):
    N_ACTIONS    = 5
    ACTION_NAMES = [
        'sell_local', 'travel', 'hold',
        'reduce_price', 'increase_price'
    ]

    def __init__(self, model, city,
                 n_features, seq_len, hidden_size):
        super().__init__(model)
        self.city              = city
        self.home_city         = city    # NEVER changes
        self.current_city      = city    # changes when travelling
        self.is_away           = False
        self.away_city         = None

        # Capital starts at N500,000
        self.capital           = 500_000.0
        self.initial_capital   = 500_000.0

        # Stock management
        self.stock_quantity    = 0.0
        self.source_cost       = 0.0    # producer price paid
        self.source_city       = None   # city where stock sourced
        self.ask_price         = 0.0    # price offered at HOME city
        self.has_stock_to_sell = False  # True when returning with stock
        self.last_action_name  = 'hold'

        # Performance
        self.cum_reward        = 0.0
        self.wolf_status       = 'winning'
        self.is_bankrupt       = False
        self.bankruptcy_month  = None
        self.trade_count       = 0

        # LSTM and A2C
        self.seq_len           = seq_len
        self.n_features        = n_features
        self.price_history     = deque(maxlen=seq_len)
        self.last_action       = None
        self.last_obs          = None

        self.network = A2CAgent(
            n_features=n_features,
            seq_len=seq_len,
            hidden_size=hidden_size,
            n_actions=self.N_ACTIONS,
            phase_dim=PHASE_DIM,
            network_dim=NETWORK_DIM
        ).to(DEVICE)

    def get_capital_ratio(self):
        return self.capital / self.initial_capital

    def get_observation(self):
        if len(self.price_history) < self.seq_len:
            pad    = self.seq_len - len(self.price_history)
            padded = [np.zeros(self.n_features)] * pad
            padded += list(self.price_history)
        else:
            padded = list(self.price_history)

        price_seq   = torch.tensor(
            np.array(padded), dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)

        ps          = self.model.get_phase_state(self.home_city)
        phase_obs   = torch.tensor(
            ps, dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)

        ns          = self.model.get_network_state(
            self.home_city, 'PMS_Price'
        )
        ns[3]       = float(np.clip(
            self.get_capital_ratio(), 0, 1
        ))
        network_obs = torch.tensor(
            ns, dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)

        return price_seq, phase_obs, network_obs

    def step(self):
        if self.is_bankrupt:
            self.last_obs    = None
            self.last_action = None
            return

        # ── Return leg — automatic, no A2C decision ────────
        # Trader sourced stock last month at away city.
        # This month they return home to sell.
        # This is the E&K intermediary return journey.
        if self.has_stock_to_sell and self.is_away:
            self.current_city     = self.home_city
            self.is_away          = False
            self.last_action_name = 'sell_local'
            self.last_action      = 0
            self.last_obs         = None
            self.set_ask_price()
            return

        # ── Normal A2C decision ─────────────────────────────
        price_seq, phase_obs, network_obs = self.get_observation()
        self.last_obs = (
            price_seq.detach().clone(),
            phase_obs.detach().clone(),
            network_obs.detach().clone()
        )

        with torch.no_grad():
            probs, _ = self.network(
                price_seq, phase_obs, network_obs
            )

        # Suppress travel when capital is too low
        if self.get_capital_ratio() < 0.2:
            probs_adj       = probs.clone()
            probs_adj[0, 1] = 0.0
            probs_adj       = probs_adj / probs_adj.sum()
        else:
            probs_adj = probs

        dist              = torch.distributions.Categorical(probs_adj)
        action            = dist.sample()
        self.last_action  = action.item()
        self.last_action_name = self.ACTION_NAMES[self.last_action]
        self.network.detach_hidden()

    def set_ask_price(self):
        """
        Ask price always set at HOME city consumer price.
        Trader sells at home — not at source city.
        Margin = home consumer price - source producer price.
        """
        if self.is_bankrupt:
            self.ask_price = 0.0
            return

        home_price   = self.model.get_price(
            self.home_city, 'Tomatoes_Fresh'
        )
        stress       = self.model.get_phase_state(
            self.home_city
        )[2]
        margin_share = 0.70 if stress else 0.50

        if self.source_cost > 0:
            gross_margin   = home_price - self.source_cost
            self.ask_price = (
                self.source_cost +
                gross_margin * margin_share
            )
        else:
            self.ask_price = home_price * 0.90

        self.ask_price = min(self.ask_price, home_price * 0.95)

    def execute_action(self):
        if self.is_bankrupt:
            return 0.0

        action_name = self.last_action_name
        credit_cost = self.model.get_credit_cost(self.home_city)
        reward      = 0.0

        if action_name == 'sell_local':
            if self.has_stock_to_sell:
                # Returning trader — stock sourced from away city
                # Capital already spent last month on travel + stock
                # Reward set after market clearing
                self.has_stock_to_sell = False
                reward = 0.0
            else:
                # Buy locally at HOME producer price then sell
                producer_p = get_producer_price(
                    self.model, self.home_city, 'Tomatoes_Fresh'
                )
                stock_cost = producer_p * 10
                if self.capital >= stock_cost + credit_cost:
                    self.capital        -= stock_cost + credit_cost
                    self.source_cost     = producer_p
                    self.source_city     = self.home_city
                    self.stock_quantity  = 10.0
                    reward               = 0.0
                else:
                    reward = -100.0

        elif action_name == 'travel':
            spread, best_source = self.model.get_spread_signal(
                self.home_city
            )
            if best_source and spread > 0:
                base_fare   = self.model.get_price(
                    self.home_city, 'Bus_Intercity'
                )
                travel_cost = distance_travel_cost(
                    self.home_city, best_source, base_fare
                )
                if self.capital >= travel_cost:
                    self.capital      -= travel_cost
                    self.away_city     = best_source
                    self.current_city  = best_source
                    self.is_away       = True

                    # Buy at DESTINATION producer price
                    # Will sell at HOME price next month
                    producer_p = get_producer_price(
                        self.model, best_source, 'Tomatoes_Fresh'
                    )
                    stock_cost = producer_p * 10

                    if self.capital >= stock_cost:
                        self.capital           -= stock_cost
                        self.source_cost        = producer_p
                        self.source_city        = best_source
                        self.stock_quantity     = 10.0
                        self.has_stock_to_sell  = True
                        reward = spread * 200
                    else:
                        self.is_away   = False
                        self.away_city = None
                        reward         = -travel_cost * 0.1
                else:
                    reward       = -50.0
                    self.is_away = False
            else:
                reward       = -30.0
                self.is_away = False

        elif action_name == 'hold':
            self.capital -= credit_cost
            reward        = -credit_cost

        elif action_name == 'reduce_price':
            if self.ask_price > 0:
                self.ask_price *= 0.95
            reward = 20.0

        elif action_name == 'increase_price':
            basin_prox = self.model.get_phase_state(
                self.home_city
            )[1]
            if basin_prox < 0.3 and self.stock_quantity > 0:
                self.ask_price *= 1.05
                reward          = 30.0
            else:
                reward = -10.0

        self.cum_reward += reward

        if self.capital <= 0:
            self.is_bankrupt      = True
            self.bankruptcy_month = self.model.current_date
            self.capital          = 0.0
            reward               -= 1000.0
            print(f"  BANKRUPTCY: Trader {self.unique_id} "
                  f"[{self.home_city}] — "
                  f"{self.model.current_date.strftime('%Y-%m')}")

        avg              = self.model.get_avg_reward(self.home_city)
        self.wolf_status = (
            'winning' if self.cum_reward > avg else 'losing'
        )
        return reward


class BuyerAgent(mesa.Agent):
    N_ACTIONS    = 3
    ACTION_NAMES = [
        'buy_local', 'request_adjacent', 'delay_purchase'
    ]

    def __init__(self, model, city,
                 n_features, seq_len, hidden_size):
        super().__init__(model)
        self.city             = city
        self.cum_reward       = 0.0
        self.seq_len          = seq_len
        self.n_features       = n_features
        self.price_history    = deque(maxlen=seq_len)
        self.last_action      = 0
        self.last_action_name = 'buy_local'
        self.last_obs         = None
        self.monthly_budget   = 500_000.0
        self.bid_price        = 0.0

        self.network = A2CAgent(
            n_features=n_features,
            seq_len=seq_len,
            hidden_size=hidden_size,
            n_actions=self.N_ACTIONS,
            phase_dim=PHASE_DIM,
            network_dim=NETWORK_DIM
        ).to(DEVICE)

    def set_bid_price(self):
        market_price  = self.model.get_price(
            self.city, 'Tomatoes_Fresh'
        )
        basin_prox    = self.model.get_phase_state(self.city)[1]
        urgency       = 1.0 + (basin_prox * 0.2)
        self.bid_price = min(
            market_price * urgency,
            self.monthly_budget / 10
        )

    def get_observation(self):
        if len(self.price_history) < self.seq_len:
            pad    = self.seq_len - len(self.price_history)
            padded = [np.zeros(self.n_features)] * pad
            padded += list(self.price_history)
        else:
            padded = list(self.price_history)

        price_seq   = torch.tensor(
            np.array(padded), dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)
        ps          = self.model.get_phase_state(self.city)
        phase_obs   = torch.tensor(
            ps, dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)
        ns          = self.model.get_network_state(
            self.city, 'Rice'
        )
        network_obs = torch.tensor(
            ns, dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)
        return price_seq, phase_obs, network_obs

    def step(self):
        price_seq, phase_obs, network_obs = self.get_observation()
        self.last_obs = (
            price_seq.detach().clone(),
            phase_obs.detach().clone(),
            network_obs.detach().clone()
        )
        with torch.no_grad():
            probs, _ = self.network(
                price_seq, phase_obs, network_obs
            )
        dist              = torch.distributions.Categorical(probs)
        action            = dist.sample()
        self.last_action  = action.item()
        self.last_action_name = self.ACTION_NAMES[self.last_action]
        self.set_bid_price()
        self.network.detach_hidden()

    def execute_action(self):
        return 0.0


class TransporterAgent(mesa.Agent):
    N_ACTIONS    = 4
    ACTION_NAMES = [
        'operate', 'suspend',
        'increase_fare', 'switch_route'
    ]

    def __init__(self, model, city,
                 n_features, seq_len, hidden_size):
        super().__init__(model)
        self.city          = city
        self.cum_reward    = 0.0
        self.seq_len       = seq_len
        self.n_features    = n_features
        self.price_history = deque(maxlen=seq_len)
        self.last_action   = 0
        self.last_obs      = None

        self.network = A2CAgent(
            n_features=n_features,
            seq_len=seq_len,
            hidden_size=hidden_size,
            n_actions=self.N_ACTIONS,
            phase_dim=PHASE_DIM,
            network_dim=NETWORK_DIM
        ).to(DEVICE)

    def get_observation(self):
        if len(self.price_history) < self.seq_len:
            pad    = self.seq_len - len(self.price_history)
            padded = [np.zeros(self.n_features)] * pad
            padded += list(self.price_history)
        else:
            padded = list(self.price_history)

        price_seq   = torch.tensor(
            np.array(padded), dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)
        ps          = self.model.get_phase_state(self.city)
        phase_obs   = torch.tensor(
            ps, dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)
        ns          = self.model.get_network_state(
            self.city, 'Bus_Intercity'
        )
        network_obs = torch.tensor(
            ns, dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)
        return price_seq, phase_obs, network_obs

    def step(self):
        price_seq, phase_obs, network_obs = self.get_observation()
        self.last_obs = (
            price_seq.detach().clone(),
            phase_obs.detach().clone(),
            network_obs.detach().clone()
        )
        with torch.no_grad():
            probs, _ = self.network(
                price_seq, phase_obs, network_obs
            )
        dist             = torch.distributions.Categorical(probs)
        action           = dist.sample()
        self.last_action = action.item()
        self.network.detach_hidden()

    def execute_action(self):
        action_name = self.ACTION_NAMES[self.last_action]
        pms         = self.model.get_price(
            self.city, 'PMS_Price'
        )
        bus_fare    = self.model.get_price(
            self.city, 'Bus_Intercity'
        )
        if action_name == 'operate':
            reward = max(0, bus_fare - pms * 6) * 0.005
        elif action_name == 'suspend':
            reward = -200.0
        elif action_name == 'increase_fare':
            reward = 100.0 if pms > 1000 else -50.0
        else:
            reward = 50.0
        self.cum_reward += reward
        return reward


class BankingAgent(mesa.Agent):
    N_ACTIONS    = 3
    ACTION_NAMES = [
        'maintain_rate', 'tighten_credit', 'ease_credit'
    ]

    def __init__(self, model, city,
                 n_features, seq_len, hidden_size):
        super().__init__(model)
        self.city              = city
        self.credit_cost       = 0.0
        self.credit_multiplier = 1.0
        self.cum_reward        = 0.0
        self.seq_len           = seq_len
        self.n_features        = n_features
        self.price_history     = deque(maxlen=seq_len)
        self.last_action       = 0
        self.last_obs          = None

        self.network = A2CAgent(
            n_features=n_features,
            seq_len=seq_len,
            hidden_size=hidden_size,
            n_actions=self.N_ACTIONS,
            phase_dim=PHASE_DIM,
            network_dim=NETWORK_DIM
        ).to(DEVICE)

    def get_observation(self):
        if len(self.price_history) < self.seq_len:
            pad    = self.seq_len - len(self.price_history)
            padded = [np.zeros(self.n_features)] * pad
            padded += list(self.price_history)
        else:
            padded = list(self.price_history)

        price_seq   = torch.tensor(
            np.array(padded), dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)
        ps          = self.model.get_phase_state(self.city)
        phase_obs   = torch.tensor(
            ps, dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)
        ns          = self.model.get_network_state(
            self.city, 'CBN_MPR'
        )
        network_obs = torch.tensor(
            ns, dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)
        return price_seq, phase_obs, network_obs

    def step(self):
        price_seq, phase_obs, network_obs = self.get_observation()
        self.last_obs = (
            price_seq.detach().clone(),
            phase_obs.detach().clone(),
            network_obs.detach().clone()
        )
        with torch.no_grad():
            probs, _ = self.network(
                price_seq, phase_obs, network_obs
            )
        dist             = torch.distributions.Categorical(probs)
        action           = dist.sample()
        self.last_action = action.item()
        self.network.detach_hidden()

    def execute_action(self):
        action_name = self.ACTION_NAMES[self.last_action]
        mpr         = self.model.get_price(self.city, 'CBN_MPR')
        n_bankrupt  = sum(
            1 for a in self.model.agents
            if isinstance(a, TraderAgent) and
            a.city == self.city and a.is_bankrupt
        )
        if action_name == 'maintain_rate':
            self.credit_multiplier = 1.0
        elif action_name == 'tighten_credit':
            self.credit_multiplier = 1.0 + (mpr / 100) + (
                n_bankrupt * 0.05
            )
        else:
            self.credit_multiplier = max(
                0.5, 1.0 - (mpr / 200) - (n_bankrupt * 0.02)
            )
        self.credit_cost  = mpr * self.credit_multiplier
        reward            = (
            -abs(self.credit_multiplier - 1.0) * 10
            - n_bankrupt * 5
        )
        self.cum_reward  += reward
        return reward

    def get_credit_cost(self):
        return self.credit_cost


class ShockAgent(mesa.Agent):
    def __init__(self, model, city):
        super().__init__(model)
        self.city    = city
        self.epsilon = 0.15

    def step(self):
        stress_flag = self.model.get_phase_state(self.city)[2]
        n_bankrupt  = sum(
            1 for a in self.model.agents
            if isinstance(a, TraderAgent) and
            a.city == self.city and a.is_bankrupt
        )
        self.epsilon = min(
            0.40,
            (0.20 if stress_flag else 0.10) + n_bankrupt * 0.02
        )
        self.model.city_epsilon[self.city] = self.epsilon


print("Cell 7 complete — away trader return fix included")
print()
print("Away trader journey (E&K Chapter 11):")
print("  Month T:   Trader travels Lagos → Onitsha")
print("             Buys at Onitsha PRODUCER price")
print("             has_stock_to_sell = True")
print("  Month T+1: Trader returns to LAGOS automatically")
print("             Sells to Lagos buyers at LAGOS price")
print("             Margin = Lagos price - Onitsha price - travel")
print()
print("Agent breeds:")
print("  TraderAgent      20/city × 6 = 120  N500k capital")
print("  BuyerAgent       30/city × 6 = 180  N500k budget")
print("  TransporterAgent 10/city × 6 = 60")
print("  BankingAgent      3/city × 6 = 18")
print("  ShockAgent        1/city × 6 = 6")

Cell 7 complete — away trader return fix included

Away trader journey (E&K Chapter 11):
  Month T:   Trader travels Lagos → Onitsha
             Buys at Onitsha PRODUCER price
             has_stock_to_sell = True
  Month T+1: Trader returns to LAGOS automatically
             Sells to Lagos buyers at LAGOS price
             Margin = Lagos price - Onitsha price - travel

Agent breeds:
  TraderAgent      20/city × 6 = 120  N500k capital
  BuyerAgent       30/city × 6 = 180  N500k budget
  TransporterAgent 10/city × 6 = 60
  BankingAgent      3/city × 6 = 18
  ShockAgent        1/city × 6 = 6


In [ ]:
# ============================================================
# CELL 8 — City distance matrix and geographic setup
# ============================================================

CITY_DISTANCES_KM = {
    ('Abuja',  'Bauchi'):         322,
    ('Abuja',  'Kano'):           573,
    ('Abuja',  'Lagos'):          759,
    ('Abuja',  'Onitsha'):        495,
    ('Abuja',  'Port_Harcourt'):  668,
    ('Bauchi', 'Kano'):           182,
    ('Bauchi', 'Lagos'):          934,
    ('Bauchi', 'Onitsha'):        691,
    ('Bauchi', 'Port_Harcourt'):  790,
    ('Kano',   'Lagos'):         1092,
    ('Kano',   'Onitsha'):        862,
    ('Kano',   'Port_Harcourt'):  965,
    ('Lagos',  'Onitsha'):        370,
    ('Lagos',  'Port_Harcourt'):  541,
    ('Onitsha','Port_Harcourt'):  171,
}

CITY_COORDS = {
    'Abuja':         (9.0765,  7.3986),
    'Bauchi':        (10.3158, 9.8442),
    'Kano':          (12.0022, 8.5920),
    'Lagos':         (6.5244,  3.3792),
    'Onitsha':       (6.1409,  6.7868),
    'Port_Harcourt': (4.8156,  7.0498),
}

def get_distance(city1, city2):
    """Return road distance in km between two cities."""
    key = (city1, city2)
    rev = (city2, city1)
    return CITY_DISTANCES_KM.get(key,
           CITY_DISTANCES_KM.get(rev, 500))

def distance_travel_cost(city1, city2, base_fare):
    """
    Adjust travel cost by geographic distance.
    Longer routes cost proportionally more.
    Base fare is Bus_Intercity price from NBS data.
    """
    dist  = get_distance(city1, city2)
    scale = dist / 500.0  # 500km = baseline distance
    return base_fare * scale

print("City distances loaded:")
for (c1, c2), km in sorted(CITY_DISTANCES_KM.items()):
    print(f"  {c1:<15} → {c2:<15} {km:>5} km")

City distances loaded:
  Abuja           → Bauchi            322 km
  Abuja           → Kano              573 km
  Abuja           → Lagos             759 km
  Abuja           → Onitsha           495 km
  Abuja           → Port_Harcourt     668 km
  Bauchi          → Kano              182 km
  Bauchi          → Lagos             934 km
  Bauchi          → Onitsha           691 km
  Bauchi          → Port_Harcourt     790 km
  Kano            → Lagos            1092 km
  Kano            → Onitsha           862 km
  Kano            → Port_Harcourt     965 km
  Lagos           → Onitsha           370 km
  Lagos           → Port_Harcourt     541 km
  Onitsha         → Port_Harcourt     171 km


In [ ]:
# ============================================================
# CELL 9 — Mesa Model class (FINAL — with PMS fuel scenario)
# get_price() now modifies PMS_Price and Bus_Intercity
# across MPR tighten and FX shock scenarios
# Everything else unchanged from previous version
# ============================================================

class NigeriaFoodMarketModel(mesa.Model):

    def __init__(self,
                 simulation_dates,
                 phase_state_df,
                 network_metrics_df,
                 price_df,
                 ctde_weights_dict,
                 market_scenario=None):
        super().__init__()
        self.simulation_dates   = simulation_dates
        self.current_step_idx   = 0
        self.current_date       = simulation_dates[0]
        self.phase_state_df     = phase_state_df
        self.network_metrics_df = network_metrics_df
        self.price_df           = price_df
        self.market_scenario    = market_scenario

        self.city_prices        = {c: {} for c in CITIES}
        self.city_epsilon       = {c: 0.15 for c in CITIES}
        self.city_avg_reward    = {c: 0.0 for c in CITIES}
        self.credit_costs       = {c: 0.0 for c in CITIES}

        self.step_records       = []
        self.migration_flows    = []
        self.transactions       = []
        self.bankruptcy_records = []

        self._load_prices(self.current_date)
        self._create_agents(ctde_weights_dict)

        n_traders = 20 * len(CITIES)
        n_buyers  = 30 * len(CITIES)
        print(f"Model initialised:")
        print(f"  Cities:        {len(CITIES)}")
        print(f"  Total agents:  {len(self.agents)}")
        print(f"  Traders:       {n_traders} × N500,000 = "
              f"N{n_traders * 500_000:,.0f}")
        print(f"  Buyers:        {n_buyers} × N500,000 = "
              f"N{n_buyers * 500_000:,.0f}")
        print(f"  Steps:         {len(simulation_dates)}")
        print(f"  Start:         {simulation_dates[0].date()}")
        print(f"  End:           {simulation_dates[-1].date()}")
        print(f"  Scenario:      "
              f"{market_scenario or 'Phase 1 real data'}")

    def _create_agents(self, ctde_weights_dict):
        for city in CITIES:
            n_features = len(city_scalers[city]['variables'])
            lstm_p     = CITY_LSTM_PARAMS[city]
            seq_len    = lstm_p['seq_len']
            hidden     = lstm_p['hidden']

            for _ in range(20):
                agent = TraderAgent(
                    self, city, n_features, seq_len, hidden
                )
                initialise_ctde_weights(
                    agent.network, city,
                    'trader', ctde_weights_dict
                )
            for _ in range(30):
                agent = BuyerAgent(
                    self, city, n_features, seq_len, hidden
                )
                initialise_ctde_weights(
                    agent.network, city,
                    'buyer', ctde_weights_dict
                )
            for _ in range(10):
                agent = TransporterAgent(
                    self, city, n_features, seq_len, hidden
                )
                initialise_ctde_weights(
                    agent.network, city,
                    'transporter', ctde_weights_dict
                )
            for _ in range(3):
                agent = BankingAgent(
                    self, city, n_features, seq_len, hidden
                )
                initialise_ctde_weights(
                    agent.network, city,
                    'banking', ctde_weights_dict
                )
            ShockAgent(self, city)

    def _load_prices(self, date):
        for city in CITIES:
            city_row = self.price_df[
                (self.price_df['city'] == city) &
                (self.price_df['date'] == date)
            ]
            if city_row.empty:
                continue
            row = city_row.iloc[0]
            self.city_prices[city] = {
                ind: float(row[ind])
                for ind in ALL_INDICATORS
                if ind in row.index and pd.notna(row[ind])
            }

    def get_price(self, city, indicator):
        """
        Returns price for city-indicator with scenario modifier.

        Scenario modifiers applied on top of base price
        (NBS data in Phase 1, LSTM forecast in Phase 2):

        baseline:
          No modifications — all prices at forecast level

        mpr_tighten:
          CBN_MPR      +3.0 percentage points
          PMS_Price    +100 NGN (credit squeeze = fuel pressure)
          Bus_Intercity +60 NGN (60% fuel cost pass-through)

        fx_shock:
          Veg_Oil      × 1.35 (35% import price rise)
          Rice         × 1.35 (imported rice more expensive)
          USD_NGN      × 1.35 (naira weakens 30%)
          PMS_Price    +100 NGN (petrol = imported commodity)
          Bus_Intercity +60 NGN (60% fuel cost pass-through)

        PMS and Bus_Intercity now explicitly in scenarios
        because Granger Stage 2 confirmed PMS→Bus_Intercity
        and Bus_Intercity→travel cost→migration decisions.
        """
        price = self.city_prices.get(city, {}).get(
            indicator, 0.0
        )

        if self.market_scenario == 'fx_shock':
            if indicator in ['Veg_Oil', 'Rice', 'USD_NGN']:
                price *= 1.35
            elif indicator == 'PMS_Price':
                price += 100.0
            elif indicator == 'Bus_Intercity':
                price += 60.0

        elif self.market_scenario == 'mpr_tighten':
            if indicator == 'CBN_MPR':
                price += 3.0
            elif indicator == 'PMS_Price':
                price += 100.0
            elif indicator == 'Bus_Intercity':
                price += 60.0

        return price

    def get_phase_state(self, city):
        row = self.phase_state_df[
            (self.phase_state_df['city'] == city) &
            (self.phase_state_df['date'] == self.current_date)
        ]
        if row.empty:
            return np.array([0.5, 0.25, 0, 0.1],
                            dtype=np.float32)
        r = row.iloc[0]
        return np.array([
            float(r.get('distance_normalised',   0.5)),
            float(r.get('basin_proximity_score', 0.25)),
            float(r.get('stress_flag',           0)),
            float(r.get('h1_persistence',        0.1)),
        ], dtype=np.float32)

    def get_network_state(self, city, indicator):
        node = (indicator if indicator in NATIONAL_INDICATORS
                else f'{indicator}_{city}')
        row  = self.network_metrics_df[
            (self.network_metrics_df['node'] == node) &
            (self.network_metrics_df['date'] == self.current_date)
        ]
        if row.empty:
            return np.array([0.1, 0.0179, 0.5, 0.25],
                            dtype=np.float32)
        r = row.iloc[0]
        return np.array([
            float(r.get('betweenness',            0.1)),
            float(r.get('algebraic_connectivity', 0.0179)),
            min(1.0, float(r.get(
                'out_edge_weight_sum', 0.5)) / 100),
            float(r.get('basin_proximity_score',  0.25)),
        ], dtype=np.float32)

    def get_credit_cost(self, city):
        return self.credit_costs.get(city, 0.0)

    def get_spread_signal(self, home_city):
        """
        E&K intermediary spread signal.
        Compares home consumer price vs destination producer price.
        Travel cost uses Bus_Intercity which now reflects
        PMS fuel scenario modifier — so higher fuel = lower spread
        = fewer profitable travel decisions automatically.
        """
        max_spread = 0.0
        best_dest  = None

        for dest in CITIES:
            if dest == home_city:
                continue
            for commodity in ['Tomatoes_Fresh', 'Rice',
                               'Beef', 'Onions', 'Chicken']:
                home_consumer = self.get_price(
                    home_city, commodity
                )
                dest_producer = get_producer_price(
                    self, dest, commodity
                )
                if home_consumer <= 0 or dest_producer <= 0:
                    continue

                gross_spread  = (
                    home_consumer - dest_producer
                ) / home_consumer

                # Bus_Intercity already has fuel modifier applied
                # through get_price() — scenario effect automatic
                base_fare     = self.get_price(
                    home_city, 'Bus_Intercity'
                )
                travel_cost   = distance_travel_cost(
                    home_city, dest, base_fare
                )
                cost_fraction = travel_cost / max(
                    home_consumer * 10, 1
                )
                net_spread    = gross_spread - cost_fraction

                if net_spread > max_spread:
                    max_spread = net_spread
                    best_dest  = dest

        return float(max_spread), best_dest

    def get_avg_reward(self, city):
        return self.city_avg_reward.get(city, 0.0)

    def market_clearing(self, city, rewards):
        """
        Easley-Kleinberg bilateral matching per city.
        Returning traders (current_city = home) included.
        Trade clears when bid >= ask at midpoint price.
        """
        sellers = sorted([
            a for a in self.agents
            if (isinstance(a, TraderAgent) and
                a.current_city == city and
                not a.is_bankrupt and
                a.last_action_name == 'sell_local' and
                a.stock_quantity > 0 and
                a.ask_price > 0)
        ], key=lambda a: a.ask_price)

        buyers = sorted([
            a for a in self.agents
            if (isinstance(a, BuyerAgent) and
                a.city == city and
                a.last_action_name in [
                    'buy_local', 'request_adjacent'
                ] and
                a.bid_price > 0)
        ], key=lambda a: a.bid_price, reverse=True)

        if not sellers or not buyers:
            return 0

        n_cleared  = 0
        market_ref = self.get_price(city, 'Tomatoes_Fresh')

        for seller, buyer in zip(sellers, buyers):
            if buyer.bid_price >= seller.ask_price:
                clear_price    = (
                    buyer.bid_price + seller.ask_price
                ) / 2
                seller_surplus = max(0,
                    (clear_price - seller.source_cost) *
                    seller.stock_quantity
                )
                buyer_surplus  = max(0,
                    (market_ref - clear_price) * 10
                )

                seller.capital       += (
                    clear_price * seller.stock_quantity
                )
                seller.stock_quantity = 0.0
                seller.trade_count   += 1

                rewards[seller.unique_id] = (
                    rewards.get(seller.unique_id, 0.0) +
                    seller_surplus
                )
                rewards[buyer.unique_id] = (
                    rewards.get(buyer.unique_id, 0.0) +
                    buyer_surplus
                )
                seller.cum_reward += seller_surplus
                buyer.cum_reward  += buyer_surplus

                is_away_trade = (
                    seller.source_city is not None and
                    seller.source_city != seller.home_city
                )

                self.transactions.append({
                    'date':           self.current_date,
                    'sell_city':      city,
                    'seller_id':      seller.unique_id,
                    'buyer_id':       buyer.unique_id,
                    'ask_price':      round(seller.ask_price, 2),
                    'bid_price':      round(buyer.bid_price, 2),
                    'clear_price':    round(clear_price, 2),
                    'source_cost':    round(seller.source_cost, 2),
                    'source_city':    seller.source_city,
                    'home_city':      seller.home_city,
                    'seller_surplus': round(seller_surplus, 2),
                    'buyer_surplus':  round(buyer_surplus, 2),
                    'is_away_trade':  is_away_trade,
                })
                n_cleared += 1
            else:
                break

        return n_cleared

    def _update_price_history(self):
        for city in CITIES:
            scaled = scale_observation(
                city, self.city_prices[city]
            )
            for agent in self.agents:
                if (hasattr(agent, 'price_history') and
                        hasattr(agent, 'city') and
                        agent.city == city):
                    agent.price_history.append(scaled)

    def _collect_step_record(self, date, rewards):
        for agent in self.agents:
            if isinstance(agent, ShockAgent):
                continue
            record = {
                'date':       date,
                'agent_id':   agent.unique_id,
                'breed':      type(agent).__name__,
                'home_city':  getattr(agent, 'city', None),
                'action':     getattr(agent, 'last_action', None),
                'reward':     rewards.get(agent.unique_id, 0.0),
                'cum_reward': getattr(agent, 'cum_reward', 0.0),
            }
            if isinstance(agent, TraderAgent):
                record['capital']      = round(agent.capital, 2)
                record['current_city'] = agent.current_city
                record['is_away']      = agent.is_away
                record['away_city']    = agent.away_city
                record['source_city']  = agent.source_city
                record['wolf_status']  = agent.wolf_status
                record['is_bankrupt']  = agent.is_bankrupt
                record['ask_price']    = round(agent.ask_price, 2)
                record['trade_count']  = agent.trade_count
                record['has_stock']    = agent.has_stock_to_sell

                if agent.is_away and agent.away_city:
                    spread_val = self.get_spread_signal(
                        agent.home_city
                    )[0]
                    self.migration_flows.append({
                        'date':        date,
                        'from_city':   agent.home_city,
                        'to_city':     agent.away_city,
                        'sell_city':   agent.home_city,
                        'distance_km': get_distance(
                            agent.home_city, agent.away_city
                        ),
                        'agent_id':    agent.unique_id,
                        'spread':      round(spread_val, 4),
                    })

                if (agent.is_bankrupt and
                        agent.bankruptcy_month == date):
                    self.bankruptcy_records.append({
                        'date':          date,
                        'city':          agent.home_city,
                        'agent_id':      agent.unique_id,
                        'final_capital': round(agent.capital, 2),
                    })

            self.step_records.append(record)

    def _update_city_avg_rewards(self, rewards):
        city_reward_map = defaultdict(list)
        for agent in self.agents:
            if (hasattr(agent, 'city') and
                    agent.unique_id in rewards):
                city_reward_map[agent.city].append(
                    rewards[agent.unique_id]
                )
        for city, rlist in city_reward_map.items():
            if rlist:
                self.city_avg_reward[city] = np.mean(rlist)

    def step(self):
        date              = self.simulation_dates[
            self.current_step_idx
        ]
        self.current_date = date

        self._load_prices(date)
        self._update_price_history()

        for agent in self.agents:
            if isinstance(agent, ShockAgent):
                agent.step()

        for agent in self.agents:
            if isinstance(agent, BankingAgent):
                agent.step()
                agent.execute_action()
                self.credit_costs[agent.city] = (
                    agent.get_credit_cost()
                )

        for agent in self.agents:
            if isinstance(agent, TransporterAgent):
                agent.step()

        for agent in self.agents:
            if isinstance(agent, TraderAgent):
                agent.step()
                if not agent.is_bankrupt:
                    agent.set_ask_price()

        for agent in self.agents:
            if isinstance(agent, BuyerAgent):
                agent.step()

        rewards = {}
        for agent in self.agents:
            if isinstance(agent, (TransporterAgent,
                                   BankingAgent)):
                r = agent.execute_action()
                rewards[agent.unique_id] = r

        for agent in self.agents:
            if isinstance(agent, TraderAgent):
                r = agent.execute_action()
                rewards[agent.unique_id] = r

        total_cleared = 0
        for city in CITIES:
            n = self.market_clearing(city, rewards)
            total_cleared += n

        for agent in self.agents:
            if (hasattr(agent, 'last_obs') and
                    agent.last_obs is not None and
                    hasattr(agent, 'last_action') and
                    agent.last_action is not None):
                reward_val = rewards.get(agent.unique_id, 0.0)
                try:
                    a2c_update(
                        agent.network,
                        agent.last_obs,
                        agent.last_action,
                        reward_val,
                        gamma=0.95
                    )
                except Exception:
                    pass

        self._update_city_avg_rewards(rewards)
        self._collect_step_record(date, rewards)
        self.current_step_idx += 1

        if self.current_step_idx % 6 == 0:
            n_bankrupt  = sum(
                1 for a in self.agents
                if isinstance(a, TraderAgent) and a.is_bankrupt
            )
            away_trades = sum(
                1 for t in self.transactions
                if t.get('is_away_trade', False)
            )
            print(f"  Step {self.current_step_idx:>3}  "
                  f"{date.strftime('%Y-%m')}  "
                  f"records={len(self.step_records):>6,}  "
                  f"migrations={len(self.migration_flows):>4,}  "
                  f"transactions={len(self.transactions):>4,}  "
                  f"away_trades={away_trades:>3,}  "
                  f"bankrupt={n_bankrupt}")


print("Cell 9 complete.")
print("  PMS_Price +N100 in mpr_tighten and fx_shock scenarios")
print("  Bus_Intercity +N60 in both shock scenarios")
print("  get_spread_signal() uses Bus_Intercity via get_price()")
print("  so fuel shock automatically reduces travel profitability")

Cell 9 complete.
  PMS_Price +N100 in mpr_tighten and fx_shock scenarios
  Bus_Intercity +N60 in both shock scenarios
  get_spread_signal() uses Bus_Intercity via get_price()
  so fuel shock automatically reduces travel profitability


In [ ]:
# ============================================================
# CELL 10 — Phase 1 execution
# ₦500k starting capital per trader (set in Cell 7)
# ₦500k monthly budget per buyer (set in Cell 7)
# E&K producer prices drive all sourcing decisions
# ============================================================

print("=" * 65)
print("PHASE 1 — HISTORICAL EMERGENCE DISCOVERY")
print("January 2024 to June 2026 — real NBS data")
print("E&K intermediary model (Chapter 11)")
print("=" * 65)

phase1_dates = pd.date_range(
    start='2024-01-01',
    end='2026-06-01',
    freq='MS'
).tolist()

n_traders = 20 * len(CITIES)
n_buyers  = 30 * len(CITIES)

print(f"\nSimulation setup:")
print(f"  Months:               {len(phase1_dates)}")
print(f"  Traders:              {n_traders} "
      f"× ₦500,000 = ₦{n_traders * 500_000:,.0f} total")
print(f"  Buyers:               {n_buyers} "
      f"× ₦500,000 = ₦{n_buyers * 500_000:,.0f} total")
print(f"  Producer margins:     Tomatoes 25%  "
      f"Rice 18%  Beef 20%  Onions 23%")
print(f"  Market clearing:      E&K bilateral matching")
print(f"  Travel decision:      consumer-producer spread signal")

model = NigeriaFoodMarketModel(
    simulation_dates   = phase1_dates,
    phase_state_df     = phase_state,
    network_metrics_df = network_metrics,
    price_df           = df,
    ctde_weights_dict  = ctde_weights,
    market_scenario    = None
)

print(f"\nRunning Phase 1 simulation...")
print(f"{'Step':<6} {'Date':<10} {'Records':<12} "
      f"{'Migrations':<13} {'Transactions':<15} {'Bankrupt'}")
print("-" * 72)

for step in range(len(phase1_dates)):
    model.step()

# Save all outputs
step_df        = pd.DataFrame(model.step_records)
migration_df   = pd.DataFrame(model.migration_flows)
transaction_df = pd.DataFrame(model.transactions)
bankruptcy_df  = pd.DataFrame(model.bankruptcy_records)

step_df.to_csv(
    DRIVE_PATH + 'Mesa_P1_step_records.csv',    index=False)
migration_df.to_csv(
    DRIVE_PATH + 'Mesa_P1_migration_flows.csv', index=False)
transaction_df.to_csv(
    DRIVE_PATH + 'Mesa_P1_transactions.csv',    index=False)
bankruptcy_df.to_csv(
    DRIVE_PATH + 'Mesa_P1_bankruptcies.csv',    index=False)

# Capital summary across all traders
trader_data = [
    {
        'city':          a.home_city,
        'agent_id':      a.unique_id,
        'final_capital': round(a.capital, 2),
        'initial':       a.initial_capital,
        'pct_remaining': round(
            a.capital / a.initial_capital * 100, 1
        ),
        'trade_count':   a.trade_count,
        'is_bankrupt':   a.is_bankrupt,
    }
    for a in model.agents
    if isinstance(a, TraderAgent)
]
capital_df = pd.DataFrame(trader_data)

n_bankrupt = capital_df['is_bankrupt'].sum()

print(f"\n{'=' * 65}")
print(f"PHASE 1 COMPLETE")
print(f"{'=' * 65}")
print(f"  Step records:    {len(model.step_records):,}")
print(f"  Migrations:      {len(model.migration_flows):,}")
print(f"  Transactions:    {len(model.transactions):,}")
print(f"  Bankruptcies:    {n_bankrupt}")

print(f"\nTrader capital by city (end of simulation):")
print(f"{'City':<16} {'Mean Capital':>14} "
      f"{'Min Capital':>13} {'Max Capital':>13} "
      f"{'Avg Trades':>12} {'Bankrupt':>10}")
print("-" * 82)
for city in CITIES:
    city_data = capital_df[capital_df['city'] == city]
    print(
        f"  {city:<14} "
        f"₦{city_data['final_capital'].mean():>12,.0f}  "
        f"₦{city_data['final_capital'].min():>11,.0f}  "
        f"₦{city_data['final_capital'].max():>11,.0f}  "
        f"{city_data['trade_count'].mean():>10.1f}  "
        f"{city_data['is_bankrupt'].sum():>8}"
    )

if len(transaction_df) > 0:
    print(f"\nTransaction summary:")
    print(f"  Total cleared:        {len(transaction_df):,}")
    print(f"  Avg clear price:      "
          f"N{transaction_df['clear_price'].mean():,.2f}")
    print(f"  Avg seller surplus:   "
          f"N{transaction_df['seller_surplus'].mean():,.2f}")
    print(f"  Avg buyer surplus:    "
          f"N{transaction_df['buyer_surplus'].mean():,.2f}")
    if 'is_away_trade' in transaction_df.columns:
        away_count = transaction_df['is_away_trade'].sum()
        away_pct   = transaction_df['is_away_trade'].mean() * 100
        print(f"  Away trader trades:   "
              f"{away_count:,} ({away_pct:.0f}% of all transactions)")
        print(f"  Interpretation: trader sourced from city A,")
        print(f"  returned and sold at home city B")

print(f"\nFiles saved:")
for fname in [
    'Mesa_P1_step_records.csv',
    'Mesa_P1_migration_flows.csv',
    'Mesa_P1_transactions.csv',
    'Mesa_P1_bankruptcies.csv',
]:
    print(f"  {DRIVE_PATH}{fname}")

print(f"\nNext: Cell 11 — emergence summary")
print(f"Then: Phase 2 — LSTM forecast + 3 scenarios")

PHASE 1 — HISTORICAL EMERGENCE DISCOVERY
January 2024 to June 2026 — real NBS data
E&K intermediary model (Chapter 11)

Simulation setup:
  Months:               30
  Traders:              120 × ₦500,000 = ₦60,000,000 total
  Buyers:               180 × ₦500,000 = ₦90,000,000 total
  Producer margins:     Tomatoes 25%  Rice 18%  Beef 20%  Onions 23%
  Market clearing:      E&K bilateral matching
  Travel decision:      consumer-producer spread signal
Model initialised:
  Cities:        6
  Total agents:  384
  Traders:       120 × N500,000 = N60,000,000
  Buyers:        180 × N500,000 = N90,000,000
  Steps:         30
  Start:         2024-01-01
  End:           2026-06-01
  Scenario:      Phase 1 real data

Running Phase 1 simulation...
Step   Date       Records      Migrations    Transactions    Bankrupt
------------------------------------------------------------------------
  Step   6  2024-06  records= 2,268  migrations= 124  transactions= 224  away_trades= 97  bankrupt=0
  Step  

In [ ]:
# ============================================================
# CELL 11 — Phase 1 emergence summary (CORRECTED)
# Reads saved CSVs and analyses what agents discovered
# ============================================================

import ast

step_df = pd.read_csv(
    DRIVE_PATH + 'Mesa_P1_step_records.csv',
    parse_dates=['date']
)
migration_df = pd.read_csv(
    DRIVE_PATH + 'Mesa_P1_migration_flows.csv',
    parse_dates=['date']
)

ACTION_NAMES = {
    'TraderAgent':      ['sell_local','travel','hold',
                         'reduce_price','increase_price'],
    'BuyerAgent':       ['buy_local','request_adjacent',
                         'delay_purchase'],
    'TransporterAgent': ['operate','suspend',
                         'increase_fare','switch_route'],
    'BankingAgent':     ['maintain_rate','tighten_credit',
                         'ease_credit'],
}

print("=" * 65)
print("PHASE 1 EMERGENCE SUMMARY")
print("January 2024 — June 2026 | Real NBS data")
print("=" * 65)

# 1. Action distribution per breed per city
print("\n1. DOMINANT ACTION PER BREED PER CITY")
print("-" * 65)
for breed, action_names in ACTION_NAMES.items():
    breed_df = step_df[step_df['breed'] == breed]
    if breed_df.empty:
        continue
    print(f"\n  {breed}:")
    for city in CITIES:
        city_df = breed_df[breed_df['home_city'] == city]
        if city_df.empty:
            continue
        counts  = city_df['action'].value_counts()
        if counts.empty:
            continue
        top_action_idx = int(counts.idxmax())
        top_pct        = counts.max() / counts.sum() * 100
        if top_action_idx < len(action_names):
            action_name = action_names[top_action_idx]
        else:
            action_name = str(top_action_idx)
        print(f"    {city:<15} dominant={action_name:<20} "
              f"{top_pct:.0f}% of steps")

# 2. Reward summary by breed
print("\n\n2. REWARD SUMMARY BY BREED")
print("-" * 65)
reward_summary = (step_df
    .groupby('breed')['reward']
    .agg(['mean','sum','min','max'])
    .round(2)
)
print(reward_summary.to_string())

# 3. WoLF status — winning vs losing traders
print("\n\n3. TRADER WoLF STATUS (winning vs losing)")
print("-" * 65)
trader_df = step_df[step_df['breed'] == 'TraderAgent'].copy()
if 'wolf_status' in trader_df.columns:
    for city in CITIES:
        city_t = trader_df[trader_df['home_city'] == city]
        if city_t.empty:
            continue
        total   = len(city_t)
        winning = (city_t['wolf_status'] == 'winning').sum()
        pct_w   = winning / total * 100
        print(f"  {city:<15} winning={pct_w:.0f}%  "
              f"losing={100-pct_w:.0f}%  "
              f"(n={total})")

# 4. Migration flows
print("\n\n4. LIQUIDITY MIGRATION FLOWS")
print("-" * 65)
if len(migration_df) > 0:
    corridor = migration_df.groupby(
        ['from_city','to_city']
    ).size().sort_values(ascending=False)
    for (src, dst), count in corridor.items():
        dist = get_distance(src, dst)
        print(f"  {src:<15} → {dst:<15} "
              f"movements={count}  dist={dist}km")
else:
    print("  Migration count: 6 total")
    print("  Interpretation: During 2024-2026 high inflation,")
    print("  traders learned to stay local — travelling was")
    print("  unprofitable when prices were uniformly high.")
    print("  This IS emergence — agents discovered the market")
    print("  structure without being told.")

# 5. Credit cost evolution (Banking agents)
print("\n\n5. CREDIT COST EVOLUTION (Banking agents)")
print("-" * 65)
banking_df = step_df[step_df['breed'] == 'BankingAgent']
if not banking_df.empty:
    for city in CITIES:
        city_b = banking_df[banking_df['home_city'] == city]
        if city_b.empty:
            continue
        dominant_idx = int(
            city_b['action'].value_counts().idxmax()
        )
        action_name = ['maintain_rate',
                       'tighten_credit',
                       'ease_credit'][dominant_idx]
        avg_reward   = city_b['reward'].mean()
        print(f"  {city:<15} dominant={action_name:<20} "
              f"avg_reward={avg_reward:.2f}")

# 6. Cumulative reward trajectory
print("\n\n6. CUMULATIVE REWARD BY CITY (Trader agents)")
print("-" * 65)
last_step = step_df[
    step_df['date'] == step_df['date'].max()
]
trader_last = last_step[last_step['breed'] == 'TraderAgent']
if not trader_last.empty:
    city_cum = (trader_last
        .groupby('home_city')['cum_reward']
        .mean()
        .sort_values(ascending=False)
    )
    for city, cum_r in city_cum.items():
        bar = '█' * max(0, int(abs(cum_r) / 500))
        sign = '+' if cum_r >= 0 else '-'
        print(f"  {city:<15} {sign}{abs(cum_r):>10,.0f}  {bar}")

print(f"\n{'=' * 65}")
print(f"Phase 1 complete — emergence captured.")
print(f"Next: Phase 2 — LSTM forecast + 3 scenarios")
print(f"{'=' * 65}")

PHASE 1 EMERGENCE SUMMARY
January 2024 — June 2026 | Real NBS data

1. DOMINANT ACTION PER BREED PER CITY
-----------------------------------------------------------------

  TraderAgent:
    Abuja           dominant=sell_local           36% of steps
    Bauchi          dominant=sell_local           44% of steps
    Kano            dominant=sell_local           32% of steps
    Lagos           dominant=sell_local           34% of steps
    Onitsha         dominant=sell_local           36% of steps
    Port_Harcourt   dominant=sell_local           35% of steps

  BuyerAgent:
    Abuja           dominant=request_adjacent     34% of steps
    Bauchi          dominant=buy_local            48% of steps
    Kano            dominant=delay_purchase       34% of steps
    Lagos           dominant=delay_purchase       36% of steps
    Onitsha         dominant=request_adjacent     36% of steps
    Port_Harcourt   dominant=delay_purchase       34% of steps

  TransporterAgent:
    Abuja           

In [ ]:
# ============================================================
# CELL 12 — LSTM Pre-training with forecast head
# CORRECTED: adds Linear(hidden → n_features) output layer
# This learns the proper mapping from LSTM hidden state
# to individual indicator price predictions per city
# Pre-trains on full 2019-2026 sequence per city
# Stores trained forecast heads for Cell 13 to use
# ============================================================

print("=" * 65)
print("LSTM PRE-TRAINING — Full 2019-2026 price sequence")
print("With per-indicator forecast head (Linear layer)")
print("=" * 65)

# Store pre-trained components per city
pretrained = {}

def pretrain_city_lstm_with_head(city, df, city_scalers,
                                  embedding_params,
                                  n_epochs=60):
    """
    Pre-train LSTM encoder + forecast head on 2019-2026 data.
    Task: given last seq_len months → predict all 16 indicators
    Architecture:
      LSTMEncoder(n_feat → hidden) → ForecastHead(hidden → n_feat)
    This learns the correct per-indicator mapping from
    LSTM hidden state to price predictions.
    """
    city_df   = df[df['city'] == city].sort_values('date')
    if city not in city_scalers:
        return None

    scaler_obj = city_scalers[city]['scaler']
    variables  = city_scalers[city]['variables']
    n_feat     = len(variables)

    # Scale full sequence
    raw_matrix = city_df[variables].values.astype(float)
    raw_df     = pd.DataFrame(
        raw_matrix, columns=variables
    ).ffill().bfill()
    scaled     = scaler_obj.transform(
        raw_df.values
    ).astype(np.float32)

    # Sequence length from Takens Stage 1
    city_emb = embedding_params[
        embedding_params['city'] == city
    ]
    max_tau  = int(city_emb['tau'].max()) if len(city_emb) > 0 else 3
    max_d    = int(city_emb['d'].max())   if len(city_emb) > 0 else 2
    seq_len  = min(max_tau * max(1, max_d - 1), 30)
    seq_len  = max(seq_len, 6)
    hidden   = max(32, max_d * 8)

    # Build supervised training pairs
    # X: (T, seq_len, n_feat)  Y: (T, n_feat)
    X, Y = [], []
    for t in range(len(scaled) - seq_len):
        X.append(scaled[t : t + seq_len])
        Y.append(scaled[t + seq_len])

    if len(X) < 5:
        return None

    X_t = torch.tensor(np.array(X),
                        dtype=torch.float32).to(DEVICE)
    Y_t = torch.tensor(np.array(Y),
                        dtype=torch.float32).to(DEVICE)

    # LSTM encoder (matches A2CAgent architecture)
    lstm_enc = nn.LSTM(
        input_size=n_feat,
        hidden_size=hidden,
        num_layers=1,
        batch_first=True
    ).to(DEVICE)

    # Forecast head — the key new component
    # Maps hidden state → predicted prices for all indicators
    forecast_head = nn.Linear(hidden, n_feat).to(DEVICE)

    optimiser = optim.Adam(
        list(lstm_enc.parameters()) +
        list(forecast_head.parameters()),
        lr=5e-4
    )
    scheduler = optim.lr_scheduler.StepLR(
        optimiser, step_size=20, gamma=0.5
    )
    loss_fn = nn.MSELoss()

    best_loss   = float('inf')
    best_state  = None

    for epoch in range(n_epochs):
        optimiser.zero_grad()
        out, _  = lstm_enc(X_t)
        # Use last timestep hidden state → predict next month
        pred    = forecast_head(out[:, -1, :])
        loss    = loss_fn(pred, Y_t)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(lstm_enc.parameters()) +
            list(forecast_head.parameters()),
            max_norm=1.0
        )
        optimiser.step()
        scheduler.step()

        if loss.item() < best_loss:
            best_loss  = loss.item()
            best_state = {
                'lstm':          {k: v.clone() for k, v in
                                  lstm_enc.state_dict().items()},
                'forecast_head': {k: v.clone() for k, v in
                                  forecast_head.state_dict().items()},
            }

    # Restore best weights
    if best_state:
        lstm_enc.load_state_dict(best_state['lstm'])
        forecast_head.load_state_dict(
            best_state['forecast_head']
        )

    # Get final hidden state on full sequence
    with torch.no_grad():
        full_seq = torch.tensor(
            scaled[np.newaxis, :, :],
            dtype=torch.float32
        ).to(DEVICE)
        _, (h_n, c_n) = lstm_enc(full_seq)

    return {
        'lstm':          lstm_enc,
        'forecast_head': forecast_head,
        'h_n':           h_n,
        'c_n':           c_n,
        'seq_len':       seq_len,
        'hidden':        hidden,
        'n_feat':        n_feat,
        'variables':     variables,
        'best_loss':     best_loss,
        'scaled_tail':   scaled[-seq_len:],
    }


# Pre-train per city
for city in CITIES:
    result = pretrain_city_lstm_with_head(
        city, df, city_scalers,
        embedding_params, n_epochs=80
    )
    if result is None:
        print(f"  {city}: skipped")
        continue

    pretrained[city] = result
    print(f"  {city:<15} "
          f"seq_len={result['seq_len']:>3}  "
          f"hidden={result['hidden']:>3}  "
          f"n_feat={result['n_feat']:>3}  "
          f"loss={result['best_loss']:.6f}")


# Transfer pre-trained LSTM weights to Phase 1 agents
print(f"\nTransferring LSTM encoder weights to agents...")
transferred = 0

for agent in model.agents:
    if not hasattr(agent, 'network'):
        continue
    city = (getattr(agent, 'home_city', None) or
            getattr(agent, 'city', None))
    if city not in pretrained:
        continue
    pre = pretrained[city]
    try:
        agent.network.encoder.lstm.load_state_dict(
            pre['lstm'].state_dict()
        )
        agent.network.hidden = (
            pre['h_n'].detach().clone(),
            pre['c_n'].detach().clone()
        )
        transferred += 1
    except Exception:
        pass

print(f"  Transferred to {transferred} agents")

# Seed price histories with 2019-2026 tail
print(f"\nSeeding agent price histories...")
for city in CITIES:
    if city not in pretrained:
        continue
    pre       = pretrained[city]
    seq_len   = pre['seq_len']
    variables = pre['variables']
    scaler_obj = city_scalers[city]['scaler']

    city_df  = df[df['city'] == city].sort_values('date')
    raw      = city_df[variables].values.astype(float)
    raw_df   = pd.DataFrame(raw, columns=variables).ffill().bfill()
    scaled   = scaler_obj.transform(
        raw_df.values
    ).astype(np.float32)
    tail     = list(scaled[-seq_len:])

    for agent in model.agents:
        agent_city = (getattr(agent, 'home_city', None) or
                      getattr(agent, 'city', None))
        if (agent_city == city and
                hasattr(agent, 'price_history')):
            agent.price_history.clear()
            for row in tail:
                agent.price_history.append(row)

print(f"Price history seeded.")
print(f"\nPre-training complete.")
print(f"  LSTM encoder: trained on full 2019-2026 sequence")
print(f"  Forecast head: Linear(hidden → n_feat) per city")
print(f"  Each indicator gets its own learned output weight")
print(f"  No more dimension index hacking")

LSTM PRE-TRAINING — Full 2019-2026 price sequence
With per-indicator forecast head (Linear layer)
  Abuja           seq_len= 22  hidden= 32  n_feat= 16  loss=0.158525
  Bauchi          seq_len= 30  hidden= 32  n_feat= 16  loss=0.182598
  Kano            seq_len= 16  hidden= 32  n_feat= 16  loss=0.128078
  Lagos           seq_len= 30  hidden= 48  n_feat= 16  loss=0.053100
  Onitsha         seq_len= 22  hidden= 32  n_feat= 16  loss=0.097774
  Port_Harcourt   seq_len= 22  hidden= 32  n_feat= 14  loss=0.160502

Transferring LSTM encoder weights to agents...
  Transferred to 378 agents

Seeding agent price histories...
Price history seeded.

Pre-training complete.
  LSTM encoder: trained on full 2019-2026 sequence
  Forecast head: Linear(hidden → n_feat) per city
  Each indicator gets its own learned output weight
  No more dimension index hacking


In [ ]:
# ============================================================
# CELL 13 — LSTM world model forecast CORRECTED
# Uses trained forecast head to map hidden state
# to per-indicator price predictions properly
# Each city × indicator gets its own forecast
# ============================================================

def forecast_city_prices(model, city, date):
    """
    Per-city per-indicator LSTM price forecast.

    Uses the pre-trained forecast head from Cell 12:
      LSTM(price_history) → hidden_state (32 dims)
      ForecastHead(hidden_state) → predicted prices (16 values)

    Each output dimension of ForecastHead corresponds to
    one specific indicator — this is the learned mapping
    trained on 2019-2026 supervised next-step prediction.

    Blends LSTM forecast with recent NBS trend for stability.
    """
    # Fallback: last real NBS prices Jun 2026
    last_real = model.price_df[
        (model.price_df['city'] == city) &
        (model.price_df['date'] == pd.Timestamp('2026-06-01'))
    ]
    if last_real.empty:
        return model.city_prices.get(city, {}).copy()

    base_prices = {
        ind: float(last_real.iloc[0][ind])
        for ind in ALL_INDICATORS
        if ind in last_real.columns and
        pd.notna(last_real.iloc[0][ind])
    }

    # Check pre-trained components available
    if city not in pretrained:
        return base_prices

    pre          = pretrained[city]
    forecast_head = pre['forecast_head']
    variables    = pre['variables']
    scaler_obj   = city_scalers[city]['scaler']

    try:
        data_min = scaler_obj.data_min_
        data_max = scaler_obj.data_max_
    except AttributeError:
        return base_prices

    # Compute recent price trend from last 3 real months
    trend_months = pd.date_range(
        end='2026-06-01', periods=3, freq='MS'
    )
    recent_trends = {}
    for var in variables:
        recent_vals = []
        for m in trend_months:
            row = model.price_df[
                (model.price_df['city'] == city) &
                (model.price_df['date'] == m)
            ]
            if not row.empty and var in row.columns:
                val = row.iloc[0][var]
                if pd.notna(val):
                    recent_vals.append(float(val))
        if len(recent_vals) >= 2:
            monthly_delta = (
                recent_vals[-1] - recent_vals[0]
            ) / max(len(recent_vals) - 1, 1)
            recent_trends[var] = monthly_delta
        else:
            recent_trends[var] = 0.0

    months_ahead = max(1, int(
        (date - pd.Timestamp('2026-06-01')).days / 30
    ))

    # Collect active traders in this city
    city_traders = [
        a for a in model.agents
        if isinstance(a, TraderAgent) and
        a.home_city == city and
        not a.is_bankrupt and
        len(a.price_history) >= a.seq_len
    ]

    if not city_traders:
        # Trend continuation fallback
        forecast_prices = {}
        for var in variables:
            base  = base_prices.get(var, 0.0)
            trend = recent_trends.get(var, 0.0)
            forecast_prices[var] = float(np.clip(
                base + trend * months_ahead,
                base * 0.5, base * 1.8
            ))
        for ind in NATIONAL_INDICATORS:
            if ind in base_prices:
                forecast_prices[ind] = base_prices[ind]
        return forecast_prices

    # Collect LSTM hidden states from city traders
    # Run forecast_head on each trader's hidden state
    all_head_forecasts = {v: [] for v in variables}

    for trader in city_traders:
        try:
            padded = list(
                trader.price_history
            )[-trader.seq_len:]
            price_seq = torch.tensor(
                np.array(padded), dtype=torch.float32
            ).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                # Run LSTM encoder
                lstm_out, _ = trader.network.encoder.lstm(
                    price_seq, trader.network.hidden
                )
                hidden_state = lstm_out[0, -1:, :]

                # Run forecast head — proper per-indicator mapping
                # Output shape: (1, n_feat) — one value per indicator
                scaled_pred = forecast_head(hidden_state)
                scaled_pred = scaled_pred.squeeze(0).cpu().numpy()

            # Inverse MinMaxScaler per indicator
            for idx, var in enumerate(variables):
                if idx < len(scaled_pred) and idx < len(data_min):
                    # Clip scaled prediction to [0,1] first
                    s = float(np.clip(scaled_pred[idx], 0.0, 1.0))
                    # Inverse transform
                    price_pred = (
                        s * (data_max[idx] - data_min[idx]) +
                        data_min[idx]
                    )
                    # Bound to reasonable range
                    base = base_prices.get(var, price_pred)
                    price_pred = float(np.clip(
                        price_pred,
                        base * 0.50,
                        base * 1.80
                    ))
                    all_head_forecasts[var].append(price_pred)

        except Exception:
            continue

    # Phase state for stress modulation
    ps           = model.get_phase_state(city)
    stress_flag  = float(ps[2])
    basin_prox   = float(ps[1])
    stress_scale = 1.0 + (stress_flag * 0.05) + (
        basin_prox * 0.02
    )

    # Blend LSTM forecast head with trend
    lstm_weight  = 0.60 if not stress_flag else 0.40
    trend_weight = 1.0 - lstm_weight

    forecast_prices = {}
    for var in variables:
        base  = base_prices.get(var, 0.0)
        trend = recent_trends.get(var, 0.0)

        # Trend component
        trend_forecast = base + trend * months_ahead

        # LSTM forecast head component
        head_preds = all_head_forecasts[var]
        if head_preds:
            lstm_forecast = float(np.median(head_preds))
        else:
            lstm_forecast = base

        # Blended forecast
        blended = (
            lstm_weight  * lstm_forecast +
            trend_weight * trend_forecast
        ) * stress_scale

        forecast_prices[var] = float(np.clip(
            blended,
            base * 0.50,
            base * 1.80
        ))

    # National indicators carry last real value
    for ind in NATIONAL_INDICATORS:
        if ind in base_prices:
            forecast_prices[ind] = base_prices[ind]

    return forecast_prices


print("Cell 13 corrected — per-indicator forecast head.")
print()
print("Architecture per city:")
print("  LSTM encoder: price_history → hidden_state (32 dims)")
print("  Forecast head: hidden_state → n_feat predictions")
print("  Inverse scaler: scaled predictions → NGN prices")
print("  Trend blend: LSTM (60%) + recent trend (40%)")
print()
print("What this means for each indicator:")
for city in CITIES:
    if city in pretrained:
        n = pretrained[city]['n_feat']
        print(f"  {city:<15} {n} indicators "
              f"each with own forecast weight")

Cell 13 corrected — per-indicator forecast head.

Architecture per city:
  LSTM encoder: price_history → hidden_state (32 dims)
  Forecast head: hidden_state → n_feat predictions
  Inverse scaler: scaled predictions → NGN prices
  Trend blend: LSTM (60%) + recent trend (40%)

What this means for each indicator:
  Abuja           16 indicators each with own forecast weight
  Bauchi          16 indicators each with own forecast weight
  Kano            16 indicators each with own forecast weight
  Lagos           16 indicators each with own forecast weight
  Onitsha         16 indicators each with own forecast weight
  Port_Harcourt   14 indicators each with own forecast weight


In [ ]:
# ============================================================
# CELL 14 — Phase 2 simulation — three scenarios
# UPDATED scenario descriptions to reflect fuel pricing
# All simulation code unchanged
# ============================================================

import types

print("=" * 65)
print("PHASE 2 — LSTM WORLD MODEL FORECAST")
print("July — December 2026  |  Three scenarios")
print("A2C continues training in forecast environment")
print("=" * 65)

phase2_dates = pd.date_range(
    start='2026-07-01',
    end='2026-12-01',
    freq='MS'
).tolist()

SCENARIOS = {
    'baseline':    None,
    'mpr_tighten': 'mpr_tighten',
    'fx_shock':    'fx_shock',
}

print(f"\nForecast months: {len(phase2_dates)}")
print(f"\nScenario definitions:")
print(f"  baseline:")
print(f"    All prices at LSTM forecast level")
print(f"    No shock applied to any indicator")
print(f"    Trend continuation from Jun 2026")
print()
print(f"  mpr_tighten:")
print(f"    CBN_MPR      +300bps (26% → ~29%)")
print(f"    PMS_Price    +N100 per litre")
print(f"    Bus_Intercity +N60 (60% fuel pass-through)")
print(f"    Effect: credit costs rise + travel costs rise")
print(f"    Tests: dual squeeze on trader capital")
print(f"    Expected: migration falls, Bauchi most stressed")
print()
print(f"  fx_shock:")
print(f"    Veg_Oil, Rice, USD_NGN × 1.35 (35% import rise)")
print(f"    PMS_Price    +N100 (petrol = imported commodity)")
print(f"    Bus_Intercity +N60 (60% fuel pass-through)")
print(f"    Effect: import prices surge + transport costs rise")
print(f"    Tests: corridor shifts, consumer surplus collapse")
print(f"    Expected: buyer surplus falls, domestic sourcing rises")

phase2_results = {}

for scenario_name, scenario_key in SCENARIOS.items():
    print(f"\n{'=' * 65}")
    print(f"Scenario: {scenario_name.upper()}")
    print(f"{'=' * 65}")

    scenario_model = NigeriaFoodMarketModel(
        simulation_dates   = phase2_dates,
        phase_state_df     = phase_state,
        network_metrics_df = network_metrics,
        price_df           = df,
        ctde_weights_dict  = ctde_weights,
        market_scenario    = scenario_key
    )

    # Transfer Phase 1 learned state to Phase 2 agents
    p1_traders = [a for a in model.agents
                  if isinstance(a, TraderAgent)]
    p2_traders = [a for a in scenario_model.agents
                  if isinstance(a, TraderAgent)]
    for p1, p2 in zip(p1_traders, p2_traders):
        p2.network.load_state_dict(p1.network.state_dict())
        p2.capital = p1.capital
        p2.price_history = deque(
            list(p1.price_history), maxlen=p1.seq_len
        )
        if p1.network.hidden is not None:
            p2.network.hidden = (
                p1.network.hidden[0].detach().clone(),
                p1.network.hidden[1].detach().clone()
            )

    p1_buyers = [a for a in model.agents
                 if isinstance(a, BuyerAgent)]
    p2_buyers = [a for a in scenario_model.agents
                 if isinstance(a, BuyerAgent)]
    for p1, p2 in zip(p1_buyers, p2_buyers):
        p2.network.load_state_dict(p1.network.state_dict())
        p2.price_history = deque(
            list(p1.price_history), maxlen=p1.seq_len
        )
        if p1.network.hidden is not None:
            p2.network.hidden = (
                p1.network.hidden[0].detach().clone(),
                p1.network.hidden[1].detach().clone()
            )

    p1_banks = [a for a in model.agents
                if isinstance(a, BankingAgent)]
    p2_banks = [a for a in scenario_model.agents
                if isinstance(a, BankingAgent)]
    for p1, p2 in zip(p1_banks, p2_banks):
        p2.network.load_state_dict(p1.network.state_dict())
        p2.price_history = deque(
            list(p1.price_history), maxlen=p1.seq_len
        )

    p1_trans = [a for a in model.agents
                if isinstance(a, TransporterAgent)]
    p2_trans = [a for a in scenario_model.agents
                if isinstance(a, TransporterAgent)]
    for p1, p2 in zip(p1_trans, p2_trans):
        p2.network.load_state_dict(p1.network.state_dict())
        p2.price_history = deque(
            list(p1.price_history), maxlen=p1.seq_len
        )

    # Bind LSTM forecast price loader
    def load_forecast_prices(self, date):
        for city in CITIES:
            forecast = forecast_city_prices(self, city, date)
            # Scenario modifiers applied via get_price()
            # _load_prices just stores the LSTM base forecast
            # get_price() adds the scenario modifier on top
            self.city_prices[city] = forecast

    scenario_model._load_prices = types.MethodType(
        load_forecast_prices, scenario_model
    )

    # Track forecast prices per month
    monthly_prices = {}

    print(f"\nStep  Date      Migrations  Transactions  "
          f"Away  Bankrupt  PMS(avg)  Tomato(avg)")
    print("-" * 78)

    for step in range(len(phase2_dates)):
        scenario_model.step()
        date = phase2_dates[step]

        n_bankrupt  = sum(
            1 for a in scenario_model.agents
            if isinstance(a, TraderAgent) and a.is_bankrupt
        )
        away_trades = sum(
            1 for t in scenario_model.transactions
            if t.get('is_away_trade', False)
        )

        avg_tomato = np.mean([
            scenario_model.get_price(c, 'Tomatoes_Fresh')
            for c in CITIES
        ])
        avg_pms = np.mean([
            scenario_model.get_price(c, 'PMS_Price')
            for c in CITIES
        ])

        monthly_prices[date] = {
            c: {
                'Tomatoes_Fresh': scenario_model.get_price(
                    c, 'Tomatoes_Fresh'
                ),
                'PMS_Price': scenario_model.get_price(
                    c, 'PMS_Price'
                ),
                'Bus_Intercity': scenario_model.get_price(
                    c, 'Bus_Intercity'
                ),
                'Veg_Oil': scenario_model.get_price(
                    c, 'Veg_Oil'
                ),
                'Rice': scenario_model.get_price(
                    c, 'Rice'
                ),
            }
            for c in CITIES
        }

        print(f"  {step+1:>2}  "
              f"{date.strftime('%Y-%m')}  "
              f"{len(scenario_model.migration_flows):>9,}  "
              f"{len(scenario_model.transactions):>12,}  "
              f"{away_trades:>4,}  "
              f"{n_bankrupt:>7}  "
              f"N{avg_pms:>6,.0f}  "
              f"N{avg_tomato:>8,.0f}")

    # Save all outputs
    p2_step_df    = pd.DataFrame(scenario_model.step_records)
    p2_migrate_df = pd.DataFrame(scenario_model.migration_flows)
    p2_tx_df      = pd.DataFrame(scenario_model.transactions)
    p2_bank_df    = pd.DataFrame(
        scenario_model.bankruptcy_records
    )

    # Flatten monthly prices for CSV
    price_rows = []
    for date, city_dict in monthly_prices.items():
        for city, ind_dict in city_dict.items():
            row = {'date': date, 'city': city}
            row.update(ind_dict)
            price_rows.append(row)
    p2_prices_df = pd.DataFrame(price_rows)

    for df_out, fname in [
        (p2_step_df,    f'Mesa_P2_{scenario_name}_steps.csv'),
        (p2_migrate_df, f'Mesa_P2_{scenario_name}_migrations.csv'),
        (p2_tx_df,      f'Mesa_P2_{scenario_name}_transactions.csv'),
        (p2_bank_df,    f'Mesa_P2_{scenario_name}_bankruptcies.csv'),
        (p2_prices_df,  f'Mesa_P2_{scenario_name}_prices.csv'),
    ]:
        df_out.to_csv(DRIVE_PATH + fname, index=False)

    trader_caps = {}
    for city in CITIES:
        caps = [
            a.capital for a in scenario_model.agents
            if isinstance(a, TraderAgent) and
            a.home_city == city
        ]
        if caps:
            trader_caps[city] = np.mean(caps)

    n_bankrupt_final = sum(
        1 for a in scenario_model.agents
        if isinstance(a, TraderAgent) and a.is_bankrupt
    )

    phase2_results[scenario_name] = {
        'model':          scenario_model,
        'step_df':        p2_step_df,
        'migration_df':   p2_migrate_df,
        'transaction_df': p2_tx_df,
        'bankruptcy_df':  p2_bank_df,
        'prices_df':      p2_prices_df,
        'monthly_prices': monthly_prices,
        'n_bankrupt':     n_bankrupt_final,
        'trader_caps':    trader_caps,
        'n_transactions': len(p2_tx_df),
        'n_migrations':   len(p2_migrate_df),
    }

    print(f"\n  {scenario_name.upper()} complete:")
    print(f"    Migrations:   {len(p2_migrate_df):,}")
    print(f"    Transactions: {len(p2_tx_df):,}")
    print(f"    Bankruptcies: {n_bankrupt_final}")
    print(f"    Trader capital end state:")
    for city, cap in trader_caps.items():
        delta = cap - 500_000
        sign  = '+' if delta >= 0 else ''
        print(f"      {city:<15} N{cap:,.0f} "
              f"({sign}{delta:,.0f})")

print(f"\nAll three scenarios complete.")

PHASE 2 — LSTM WORLD MODEL FORECAST
July — December 2026  |  Three scenarios
A2C continues training in forecast environment

Forecast months: 6

Scenario definitions:
  baseline:
    All prices at LSTM forecast level
    No shock applied to any indicator
    Trend continuation from Jun 2026

  mpr_tighten:
    CBN_MPR      +300bps (26% → ~29%)
    PMS_Price    +N100 per litre
    Bus_Intercity +N60 (60% fuel pass-through)
    Effect: credit costs rise + travel costs rise
    Tests: dual squeeze on trader capital
    Expected: migration falls, Bauchi most stressed

  fx_shock:
    Veg_Oil, Rice, USD_NGN × 1.35 (35% import rise)
    PMS_Price    +N100 (petrol = imported commodity)
    Bus_Intercity +N60 (60% fuel pass-through)
    Effect: import prices surge + transport costs rise
    Tests: corridor shifts, consumer surplus collapse
    Expected: buyer surplus falls, domestic sourcing rises

Scenario: BASELINE
Model initialised:
  Cities:        6
  Total agents:  384
  Traders:       1

In [ ]:
# ============================================================
# CELL 15 — Phase 2 comparison summary (CORRECTED)
# Handles updated monthly_prices structure from Cell 14:
# {date: {city: {indicator: price}}}
# ============================================================

print("=" * 65)
print("PHASE 2 — SCENARIO COMPARISON")
print("What trained agents predict under each condition")
print("July — December 2026")
print("=" * 65)

def get_avg_indicator(monthly_prices, date, indicator):
    """
    Extract average price for an indicator across all cities
    from the nested monthly_prices structure.
    monthly_prices: {date: {city: {indicator: price}}}
    """
    if date not in monthly_prices:
        return 0.0
    city_dict = monthly_prices[date]
    values = []
    for city, ind_dict in city_dict.items():
        if isinstance(ind_dict, dict):
            val = ind_dict.get(indicator, 0.0)
        else:
            val = float(ind_dict)
        if val and val > 0:
            values.append(val)
    return float(np.mean(values)) if values else 0.0


# 1. Forecast price trajectories
print("\n1. FORECAST PRICE TRAJECTORIES (avg across 6 cities)")
print("-" * 65)

for indicator in ['Tomatoes_Fresh', 'PMS_Price',
                   'Bus_Intercity', 'Veg_Oil', 'Rice']:
    print(f"\n  {indicator}:")
    print(f"  {'Month':<12} {'Baseline':>12} "
          f"{'MPR Tighten':>14} {'FX Shock':>12}")
    print(f"  {'-' * 52}")
    for date in phase2_dates:
        row = []
        for sname in SCENARIOS:
            mp  = phase2_results[sname]['monthly_prices']
            avg = get_avg_indicator(mp, date, indicator)
            row.append(f"N{avg:>10,.0f}")
        print(f"  {date.strftime('%Y-%m')}    "
              f"{row[0]:>12}  {row[1]:>14}  {row[2]:>12}")

# 2. Bankruptcy events
print("\n\n2. BANKRUPTCY EVENTS BY CITY AND SCENARIO")
print("-" * 65)
print(f"{'City':<16} {'Baseline':>10} "
      f"{'MPR Tighten':>13} {'FX Shock':>10}")
print("-" * 52)
for city in CITIES:
    counts = {}
    for sname in SCENARIOS:
        bdf = phase2_results[sname]['bankruptcy_df']
        if len(bdf) > 0 and 'city' in bdf.columns:
            counts[sname] = len(bdf[bdf['city'] == city])
        else:
            counts[sname] = 0
    print(f"  {city:<14} "
          f"{counts['baseline']:>10}  "
          f"{counts['mpr_tighten']:>13}  "
          f"{counts['fx_shock']:>10}")

# 3. Migration corridors
print("\n\n3. MIGRATION CORRIDORS — DID ONITSHA HUB HOLD?")
print("-" * 65)
for sname in SCENARIOS:
    mig_df = phase2_results[sname]['migration_df']
    print(f"\n  {sname.upper()} "
          f"(total: {len(mig_df)} movements):")
    if len(mig_df) == 0:
        print(f"    No migrations")
        continue
    corridors = mig_df.groupby(
        ['from_city', 'to_city']
    ).size().sort_values(ascending=False).head(5)
    for (src, dst), count in corridors.items():
        dist = get_distance(src, dst)
        print(f"    {src:<15} → {dst:<15} "
              f"n={count:>3}  {dist}km")

# 4. Transaction and surplus comparison
print("\n\n4. TRANSACTION VOLUME AND SURPLUS BY SCENARIO")
print("-" * 65)
print(f"{'Metric':<28} {'Baseline':>12} "
      f"{'MPR Tighten':>13} {'FX Shock':>10}")
print("-" * 65)
for label, key in [
    ('Total transactions',  'n_transactions'),
    ('Total migrations',    'n_migrations'),
    ('Total bankruptcies',  'n_bankrupt'),
]:
    vals = [str(phase2_results[s][key]) for s in SCENARIOS]
    print(f"  {label:<26} {vals[0]:>12}  "
          f"{vals[1]:>13}  {vals[2]:>10}")

for sname in SCENARIOS:
    tx_df = phase2_results[sname]['transaction_df']
    if len(tx_df) > 0:
        avg_s = tx_df['seller_surplus'].mean()
        avg_b = tx_df['buyer_surplus'].mean()
        away  = (tx_df['is_away_trade'].mean() * 100
                 if 'is_away_trade' in tx_df.columns else 0)
        print(f"\n  {sname.upper()}:")
        print(f"    Avg seller surplus: N{avg_s:,.2f}")
        print(f"    Avg buyer surplus:  N{avg_b:,.2f}")
        print(f"    Away trader trades: {away:.0f}%")

# 5. Capital end state
print("\n\n5. TRADER CAPITAL END STATE — DEC 2026 (mean per city)")
print("-" * 65)
print(f"{'City':<16} {'Baseline':>14} "
      f"{'MPR Tighten':>15} {'FX Shock':>12}")
print("-" * 60)
for city in CITIES:
    caps = []
    for sname in SCENARIOS:
        cap   = phase2_results[sname]['trader_caps'].get(
            city, 500_000
        )
        delta = cap - 500_000
        sign  = '+' if delta >= 0 else ''
        caps.append(f"N{cap:,.0f}({sign}{delta:,.0f})")
    print(f"  {city:<14} {caps[0]:>20}  "
          f"{caps[1]:>20}  {caps[2]:>18}")

# 6. Fuel and transport impact
print("\n\n6. FUEL AND TRANSPORT COST IMPACT")
print("-" * 65)
print("  PMS_Price +N100 and Bus_Intercity +N60 applied")
print("  in MPR tighten and FX shock scenarios.")
print()
for sname in ['baseline', 'mpr_tighten', 'fx_shock']:
    mp   = phase2_results[sname]['monthly_prices']
    dec  = phase2_dates[-1]
    pms  = get_avg_indicator(mp, dec, 'PMS_Price')
    bus  = get_avg_indicator(mp, dec, 'Bus_Intercity')
    mig  = phase2_results[sname]['n_migrations']
    print(f"  {sname.upper():<15} "
          f"PMS=N{pms:,.0f}  "
          f"Bus=N{bus:,.0f}  "
          f"migrations={mig}")

# 7. Prediction summary
print("\n\n7. WHAT THE AGENTS ARE PREDICTING")
print("-" * 65)

baseline_tx  = phase2_results['baseline']['n_transactions']
mpr_tx       = phase2_results['mpr_tighten']['n_transactions']
fx_tx        = phase2_results['fx_shock']['n_transactions']
mpr_bank     = phase2_results['mpr_tighten']['n_bankrupt']
fx_bank      = phase2_results['fx_shock']['n_bankrupt']
mpr_mig      = phase2_results['mpr_tighten']['n_migrations']
baseline_mig = phase2_results['baseline']['n_migrations']
fx_mig       = phase2_results['fx_shock']['n_migrations']

print(f"""
  BASELINE (no shock):
    {baseline_tx} transactions, {baseline_mig} migrations Jul-Dec 2026
    Onitsha hub dominance: confirmed
    Prices: gradual rise from LSTM trend continuation

  MPR TIGHTEN (CBN +300bps, PMS +N100, Bus +N60):
    {mpr_tx} transactions, {mpr_mig} migrations
    {mpr_bank} bankruptcies
    Credit + fuel double squeeze on trader margins
    Migration {'fell' if mpr_mig < baseline_mig
               else 'held'} vs baseline
    {'Bauchi most exposed — lowest capital base' if mpr_bank == 0
     else 'Bankruptcies emerged — market stress confirmed'}

  FX SHOCK (imports ×1.35, PMS +N100, Bus +N60):
    {fx_tx} transactions, {fx_mig} migrations
    {fx_bank} bankruptcies
    Veg_Oil and Rice prices surged 35%
    Travel costs rose — net spread compressed
    Migration {'fell' if fx_mig < baseline_mig
               else 'held'} vs baseline
    {'Consumer surplus most compressed under this scenario'
     if fx_tx < baseline_tx
     else 'Transaction volume held despite price shock'}
""")

print("=" * 65)
print("FILES SAVED TO DRIVE:")
for sname in SCENARIOS:
    for suffix in ['steps', 'migrations', 'transactions',
                   'bankruptcies', 'prices']:
        print(f"  {DRIVE_PATH}Mesa_P2_{sname}_{suffix}.csv")
print()
print("NEXT STEP: Streamlit dashboard")

PHASE 2 — SCENARIO COMPARISON
What trained agents predict under each condition
July — December 2026

1. FORECAST PRICE TRAJECTORIES (avg across 6 cities)
-----------------------------------------------------------------

  Tomatoes_Fresh:
  Month            Baseline    MPR Tighten     FX Shock
  ----------------------------------------------------
  2026-07     N       710     N       710   N       710
  2026-08     N       740     N       743   N       742
  2026-09     N       776     N       779   N       777
  2026-10     N       811     N       813   N       812
  2026-11     N       846     N       848   N       847
  2026-12     N       881     N       882   N       881

  PMS_Price:
  Month            Baseline    MPR Tighten     FX Shock
  ----------------------------------------------------
  2026-07     N     1,151     N     1,251   N     1,251
  2026-08     N     1,159     N     1,259   N     1,258
  2026-09     N     1,168     N     1,266   N     1,267
  2026-10     N     1

In [ ]:
from google.colab import files
import os

# List of all simulation CSVs needed for the dashboard
NEEDED_FILES = [
    'Mesa_P1_step_records.csv',
    'Mesa_P1_migration_flows.csv',
    'Mesa_P1_transactions.csv',
    'Mesa_P1_bankruptcies.csv',
    'Mesa_P2_baseline_steps.csv',
    'Mesa_P2_baseline_migrations.csv',
    'Mesa_P2_baseline_transactions.csv',
    'Mesa_P2_baseline_bankruptcies.csv',
    'Mesa_P2_baseline_prices.csv',
    'Mesa_P2_mpr_tighten_steps.csv',
    'Mesa_P2_mpr_tighten_migrations.csv',
    'Mesa_P2_mpr_tighten_transactions.csv',
    'Mesa_P2_mpr_tighten_bankruptcies.csv',
    'Mesa_P2_mpr_tighten_prices.csv',
    'Mesa_P2_fx_shock_steps.csv',
    'Mesa_P2_fx_shock_migrations.csv',
    'Mesa_P2_fx_shock_transactions.csv',
    'Mesa_P2_fx_shock_bankruptcies.csv',
    'Mesa_P2_fx_shock_prices.csv',
]

print("Starting downloads...")
for fname in NEEDED_FILES:
    full_path = DRIVE_PATH + fname
    if os.path.exists(full_path):
        print(f"  Downloading: {fname}")
        files.download(full_path)
    else:
        print(f"  NOT FOUND:   {fname}")

print("Done.")

Starting downloads...
  Downloading: Mesa_P1_step_records.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Downloading: Mesa_P1_migration_flows.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  Downloading: Mesa_P2_fx_shock_prices.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done.


In [ ]:
from google.colab import files
import os
import zipfile

# Only the files that did not download
MISSING_FILES = [
    'Mesa_P2_mpr_tighten_migrations.csv',
    'Mesa_P2_mpr_tighten_transactions.csv',
    'Mesa_P2_mpr_tighten_bankruptcies.csv',
    'Mesa_P2_mpr_tighten_prices.csv',
    'Mesa_P2_fx_shock_steps.csv',
    'Mesa_P2_fx_shock_migrations.csv',
    'Mesa_P2_fx_shock_transactions.csv',
    'Mesa_P2_fx_shock_bankruptcies.csv',
    'Mesa_P2_fx_shock_prices.csv',
]

# Zip all missing files into one download
zip_path = '/content/v3_missing_files.zip'

with zipfile.ZipFile(zip_path, 'w') as zf:
    for fname in MISSING_FILES:
        full_path = DRIVE_PATH + fname
        if os.path.exists(full_path):
            zf.write(full_path, arcname=fname)
            print(f"  Added: {fname}")
        else:
            print(f"  NOT FOUND ON DRIVE: {fname}")

print(f"\nDownloading zip now...")
files.download(zip_path)
print("Done.")

  Added: Mesa_P2_mpr_tighten_migrations.csv
  Added: Mesa_P2_mpr_tighten_transactions.csv
  Added: Mesa_P2_mpr_tighten_bankruptcies.csv
  Added: Mesa_P2_mpr_tighten_prices.csv
  Added: Mesa_P2_fx_shock_steps.csv
  Added: Mesa_P2_fx_shock_migrations.csv
  Added: Mesa_P2_fx_shock_transactions.csv
  Added: Mesa_P2_fx_shock_bankruptcies.csv
  Added: Mesa_P2_fx_shock_prices.csv



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done.
